# 14. 레짐 전환 방법 (Regime-Switching Methods)

14번은 하나의 모형이 아니라 **"레짐을 모형에 넣는 방법들 전체"**를 가리키는 말이야. 처음 대화에서 소개한 2025년 S&P 500 연구도 soft Markov switching, XGBoost로 예측 시점의 군집을 배정하는 분포적 스펙트럴 클러스터링, Mood 검정으로 구간을 나눈 뒤 베이지안 방법으로 군집화하는 계수 기반 방법처럼 **여러 방식**을 비교했어.

그래서 이번엔 지도를 그리는 게 목표야. 8~10번(HMM, Markov switching, JM)이 이 지도의 어디에 있는지도 자리매김할게. 순서는 이래.

1. 레짐 방법을 분류하는 두 가지 질문
2. 관측 변수 임계값 모형 (TAR)
3. 부드러운 전환 모형 (STR)
4. 잠재 레짐 모형 (HMM, MS 복습)
5. 군집 기반 모형 (k-means, JM)
6. 구조적 변화 모형 (한 번 바뀌면 돌아오지 않는 레짐)
7. 규칙 기반·제도적 레짐
8. 하드 vs 소프트 레짐
9. 레짐을 예측 모형에 넣는 세 가지 방식
10. 비교표, 네 프로젝트 적용, 코드

---

## 0단계: 모든 레짐 방법은 두 질문에 답한다

**질문 A: 레짐을 어떻게 정하나?**

- 레짐이 **관측 가능**한가(VKOSPI > 25), **숨겨져** 있나(HMM)?
- 전환이 **급격**한가(0 아니면 1), **부드러운**가(0에서 1로 서서히)?
- 레짐이 **반복**되나(평온 → 혼란 → 평온), **한 번 바뀌면 끝**인가(구조적 변화)?

**질문 B: 정한 레짐을 모형에서 어떻게 쓰나?**

- 레짐마다 모형을 따로 만드나?
- 하나의 모형에 레짐 변수를 넣나?
- 레짐과 모형을 동시에 추정하나?

아래 방법들은 전부 이 두 질문에 대한 서로 다른 답이야.

---

## 1단계: 임계값 모형 (Threshold Autoregression, TAR)

**아이디어:** 관측 가능한 변수 $q_t$(전이 변수)가 기준값 c를 넘으면 레짐이 바뀐다.

$$
y_t = \begin{cases} \alpha_1 + \beta_1 x_t + \varepsilon_t & \text{if } q_{t-1} \leq c \quad (\text{레짐 1}) \\ \alpha_2 + \beta_2 x_t + \varepsilon_t & \text{if } q_{t-1} > c \quad (\text{레짐 2}) \end{cases}
$$

지시함수로 한 줄로 쓰면:

$$
y_t = (\alpha_1 + \beta_1 x_t)\,\mathbf{1}\{q_{t-1} \leq c\} + (\alpha_2 + \beta_2 x_t)\,\mathbf{1}\{q_{t-1} > c\} + \varepsilon_t
$$

예를 들어 $q_t$ = VKOSPI라면, "변동성 지수가 c를 넘는 날에는 RSJ가 수익률에 주는 영향($\beta$)이 달라진다"는 모형이야. $q_t$가 $y_t$ 자신의 과거값이면 SETAR(Self-Exciting TAR)라고 불러.

**c는 어떻게 추정하나 (격자 탐색):** c를 고정하면 모형은 그냥 두 개의 OLS야. 그래서:

1. c 후보들을 정한다 (보통 $q_t$의 15~85 백분위 사이 값들).
2. 각 c마다 두 레짐으로 나눠 OLS를 돌리고 잔차제곱합(SSR)을 계산한다.
3. SSR이 가장 작은 c를 고른다.

$$
\hat{c} = \arg\min_{c}\ SSR(c)
$$

양 끝 15%를 빼는 이유는, 한 레짐에 표본이 너무 적으면 추정이 불가능하거나 우연히 잘 맞는 c가 뽑히기 때문이야.

**장점:** 레짐이 **관측 가능**하니까 해석이 명확하고, 실시간으로 "오늘 레짐"을 확실히 알 수 있어.
**단점:** 9번에서 본 **Davies 문제**가 그대로 있어. 귀무가설 "레짐 없음"($\alpha_1 = \alpha_2$, $\beta_1 = \beta_2$) 아래서는 c가 정의되지 않으니, 표준 검정이 작동하지 않아. Hansen(1996, 2000)의 부트스트랩 기반 검정을 써야 해.

---

## 2단계: 부드러운 전환 모형 (Smooth Transition Regression, STR)

TAR은 $q_t$가 c를 **살짝만** 넘어도 레짐이 통째로 바뀌어. VKOSPI 24.9와 25.1이 완전히 다른 세계라는 건 부자연스럽지. STR은 지시함수를 **부드러운 함수**로 바꿔.

$$
y_t = (\alpha_1 + \beta_1 x_t)\big[1 - G(q_{t-1})\big] + (\alpha_2 + \beta_2 x_t)\,G(q_{t-1}) + \varepsilon_t
$$

전이함수 G로는 로지스틱 함수를 많이 써(LSTR).

$$
G(q;\ \gamma, c) = \frac{1}{1 + e^{-\gamma(q - c)}} \in (0, 1)
$$

- **c**: 전환의 중심 (G = 0.5가 되는 지점)
- **γ**: 전환의 **가파름**

γ에 따라 G가 어떻게 되는지 숫자로 보자. c = 25라고 할게.

| $q$ (VKOSPI) | γ = 0.2 | γ = 1 | γ = 10 |
|---|---|---|---|
| 15 | 0.12 | 0.00 | 0.00 |
| 22 | 0.35 | 0.05 | 0.00 |
| 25 | 0.50 | 0.50 | 0.50 |
| 28 | 0.65 | 0.95 | 1.00 |
| 35 | 0.88 | 1.00 | 1.00 |

- **γ → ∞**: G가 계단함수가 돼서 **TAR과 같아져.**
- **γ → 0**: G가 모든 q에서 0.5 근처라 두 레짐이 반씩 섞여. 사실상 **선형 모형**이 돼.

즉 STR은 **선형 모형과 TAR을 양 끝으로 포함하는 일반화**야. 추정은 γ, c를 포함해 비선형 최소제곱으로 해.

**해석의 편리함:** 오늘의 기울기는 두 레짐 기울기의 가중평균이야.

$$
\beta_t^{\text{실효}} = \beta_1\big[1 - G(q_{t-1})\big] + \beta_2\,G(q_{t-1})
$$

7번 HARQ에서 √RQ에 따라 계수가 연속적으로 바뀌었던 것과 같은 발상이야. HARQ는 선형 함수로, STR은 로지스틱 함수로 바뀐다는 차이만 있어.

---

## 3단계: 잠재 레짐 모형 (HMM, Markov Switching) — 복습

8, 9번에서 다룬 계열이야. 지도 위의 위치만 짚을게.

- 레짐 $S_t$는 **숨겨져** 있고, 전이행렬 P에 따라 **확률적으로** 바뀌어.
- 1, 2단계와의 결정적 차이는 이거야. **레짐을 결정하는 게 관측 변수가 아니라 데이터 전체의 패턴**이야. 연구자가 "무엇이 레짐을 바꾸는가"를 미리 정하지 않아도 돼.
- 대가는 레짐을 **확률로만** 알 수 있다는 것, 그리고 스무딩과 look-ahead 문제야.
- 9번의 **TVTP**는 1단계와 3단계의 중간이야. 레짐은 숨겨져 있지만, 전환 확률은 관측 변수가 결정하니까.

---

## 4단계: 군집 기반 모형 (k-means, JM, spectral clustering)

**아이디어:** 레짐을 "특징 벡터가 서로 비슷한 날들의 묶음"으로 정의해.

- **k-means**: 시간 구조 없이 거리만 봐.
- **JM (10번)**: k-means에 전환 벌금 λ를 추가해서 지속성을 부여해.
- **Spectral clustering (13번)**: 건너뛰었지만, 한 줄로 말하면 데이터 사이의 **유사도 그래프**를 만들고 그 구조로 군집을 나누는 방법이야. 원형이나 비선형 모양의 군집도 잡을 수 있어.

확률모형(3단계)과 비교하면 **분포 가정이 없어.** 방출분포를 정규분포로 가정하는 HMM과 달리 "비슷한 것끼리 묶는다"만 해. 대신 레짐 확률 같은 확률적 해석은 기본적으로 제공하지 않아(CJM 같은 확장은 예외).

**군집 방법의 실무적 문제:** 군집화는 학습 데이터 전체에 대해 한 번에 이뤄지니, **새로운 날이 어느 군집인지 배정하는 규칙**이 따로 필요해. 앞에서 인용한 2025년 연구가 스펙트럴 클러스터링으로 군집을 만든 뒤 **XGBoost로 예측 시점의 군집을 배정**한 게 바로 이 문제를 푸는 방식이야. 군집 결과를 라벨로 삼아 분류기를 학습시키고, 새 데이터에는 분류기를 적용하는 거지.

---

## 5단계: 구조적 변화 모형 (Structural Breaks / Change-Point)

지금까지의 레짐은 **반복**됐어(평온 → 혼란 → 평온). 그런데 어떤 변화는 **한 번 일어나면 돌아오지 않아.** 예를 들어 거래시간 연장, 가격제한폭 확대(2015년 ±15% → ±30%), 호가단위 개편 같은 거야.

$$
y_t = x_t'\beta_j + \varepsilon_t, \qquad T_{j-1} < t \leq T_j, \quad j = 1, \dots, m+1
$$

m개의 **변화 시점** $T_1, \dots, T_m$이 전체 기간을 m+1개 구간으로 나누고, 구간마다 계수가 달라.

**추정 (Bai-Perron):** 변화 시점을 바꿔가며 전체 SSR이 최소가 되는 조합을 찾아. 모든 조합을 다 따지면 불가능하니까 **동적계획법**을 써. 10번 JM의 알고리즘과 같은 원리야.

**Markov switching과의 차이:**

| | Markov switching | 구조적 변화 |
|---|---|---|
| 레짐 반복 | 반복 (같은 레짐이 다시 옴) | 반복 없음 (매 구간이 새 레짐) |
| 파라미터 공유 | 같은 레짐끼리 계수 공유 | 구간마다 별개 |
| 적합한 현상 | 경기순환, 변동성 국면 | 제도 변화, 시장 구조 변화 |

**실시간 버전(online change-point detection):** CUSUM처럼 "누적 잔차가 일정 수준을 넘으면 변화 발생으로 판단"하는 방법이 있어. 실시간 감지가 목적일 때 써.

---

## 6단계: 규칙 기반·제도적 레짐

추정 없이 **미리 정한 규칙**으로 레짐을 정의하는 방법이야.

- **기술적 규칙:** KOSPI가 200일 이동평균 아래면 약세 레짐
- **고정 임계값:** VKOSPI > 25면 고변동 레짐 (1단계와 달리 c를 추정하지 않고 정함)
- **외부 기준:** 경기침체 공식 기간
- **제도적 레짐:** 공매도 금지 기간, 가격제한폭 변경 전후

**장점:** 투명하고, 추정 오차가 없고, look-ahead 위험이 거의 없어(규칙을 사전에 정했다면).
**단점:** 임계값이 자의적이야. 25를 왜 골랐냐고 물으면 답하기 어려워. 그리고 여러 임계값을 시도해보고 결과가 좋은 걸 고르면, 12번에서 본 **선택 편향**이 그대로 생겨.

**제도적 레짐은 특별해.** 공매도 금지는 시장 데이터를 보고 정한 게 아니라 **정책 결정**으로 외부에서 주어진 거야. 그래서 1~5단계의 모든 통계적 레짐이 가진 문제, 즉 "데이터로 레짐을 정하고 같은 데이터로 레짐 효과를 검정하는 순환성"에서 자유로워. 처음 대화부터 계속 이걸 강조한 이유야.

---

## 7단계: 하드 레짐 vs 소프트 레짐

레짐 정보를 **0/1로 쓰느냐, 확률로 쓰느냐**의 문제야.

- **하드:** $D_t = \mathbf{1}\{\text{레짐 2}\}$. TAR, k-means, JM, 규칙 기반이 여기에 속해.
- **소프트:** $\pi_t = P(S_t = 2 \mid \text{정보}_t) \in [0, 1]$. HMM, MS, STR의 G, CJM이 여기에 속해.

**예측할 때 소프트가 원칙적으로 더 나은 이유 (유도):** 전체 기댓값의 법칙(law of total expectation)에 따라,

$$
E[y_{t+1} \mid \text{정보}_t] = \sum_k P(S_{t+1} = k \mid \text{정보}_t)\cdot E[y_{t+1} \mid S_{t+1} = k,\ \text{정보}_t]
$$

즉 최적 예측은 **레짐별 예측을 레짐 확률로 가중평균**한 거야. 하드 레짐은 이 확률을 가장 높은 쪽 1, 나머지 0으로 반올림하는 셈이야. 레짐 확률이 0.55 대 0.45처럼 애매한 날에는 정보를 크게 잃어.

**숫자 예시:** 레짐 1 예측 +0.3%, 레짐 2 예측 −0.5%, 레짐 확률이 (0.55, 0.45)라면

- 소프트: $0.55 \times 0.3 + 0.45 \times (-0.5) = -0.06\%$
- 하드: 레짐 1로 반올림해서 $+0.3\%$

방향까지 뒤집혀.

**하드가 나을 때도 있어:** 매매처럼 결정 자체가 이산적일 때(들어가거나 나가거나), 또는 확률 추정이 부정확해서 오히려 노이즈만 더할 때야. 10번에서 Shu et al.이 0/1 전략에서는 JM과 CJM의 차이가 크지 않았다고 한 게 이 경우야.

---

## 8단계: 레짐을 예측 모형에 넣는 세 가지 방식 (질문 B)

네 프로젝트에 가장 직접적으로 중요한 부분이야.

**방식 ① 표본 분할:** 레짐 1인 날들과 레짐 2인 날들로 데이터를 나눠 각각 회귀를 돌려.

$$
y_t = \alpha_1 + \beta_1 x_t + \varepsilon_t \quad (\text{레짐 1 표본}), \qquad y_t = \alpha_2 + \beta_2 x_t + \varepsilon_t \quad (\text{레짐 2 표본})
$$

직관적이지만, 표본이 쪼개져서 검정력이 떨어지고, "$\beta_1$과 $\beta_2$가 **통계적으로 다른가**"를 직접 검정하기 불편해.

**방식 ② 상호작용항 (추천):** 하나의 회귀에 레짐 변수와의 곱을 넣어.

$$
\boxed{y_t = \alpha + \beta\,x_t + \gamma\,D_t + \delta\,(x_t \times D_t) + \varepsilon_t}
$$

$D_t$에 레짐 2 더미를 넣고 레짐별로 풀어보면:

- 레짐 1 ($D_t = 0$): $y_t = \alpha + \beta\,x_t$
- 레짐 2 ($D_t = 1$): $y_t = (\alpha + \gamma) + (\beta + \delta)\,x_t$

즉 **δ가 곧 "레짐 2에서 기울기가 얼마나 다른가"**야. 네 연구 질문 "레짐별로 시그널이 다른가?"가 **$H_0: \delta = 0$이라는 t-검정 하나**로 바뀌어. $D_t$ 대신 소프트 확률 $\pi_t$를 넣어도 똑같이 작동해.

모든 계수에 상호작용을 넣으면 방식 ①과 계수 추정치가 같아지지만, 방식 ②는 잔차 분산을 공유하고 차이를 바로 검정할 수 있어서 실무적으로 더 편해. 5번 IVOL이나 4번 semibeta 분석에서 Fama-MacBeth 회귀를 할 때도 이 구조를 그대로 쓰면 돼.

**방식 ③ 동시 추정:** 9번의 Markov switching 회귀처럼 레짐과 계수를 한 번에 추정해. 가장 우아하지만, 9번에서 말했듯 **관계로 레짐을 정의하니 순환성**이 있어.

---

## 9단계: 전체 비교표

| 방법 | 레짐 관측 | 전환 형태 | 반복 | 추정 대상 | look-ahead 위험 | 해석 |
|---|---|---|---|---|---|---|
| 규칙·제도 | O | 급격 | 둘 다 | 없음 | 낮음 | 매우 쉬움 |
| TAR | O | 급격 | O | c, 계수 | c를 전체 표본으로 추정하면 있음 | 쉬움 |
| STR | O | 부드러움 | O | γ, c, 계수 | 위와 같음 | 쉬움 |
| HMM / MS | X | 확률적 | O | P, 분포/계수 | 스무딩 쓰면 큼 | 보통 |
| k-means / JM | X | 급격 | O | 중심, (λ) | 전체 경로 재계산 시 있음 | 보통 |
| 구조적 변화 | X (시점 추정) | 급격 | X | 변화 시점, 계수 | 사후 추정이면 큼 | 쉬움 |

"look-ahead 위험" 열을 봐. **추정이 들어가는 모든 방법은 전체 표본으로 추정하는 순간 미래 정보를 쓴 거야.** 16번에서 이 문제의 해결법을 다룰 거야.

---

## 10단계: 네 프로젝트에 어떻게 적용할까

**여러 방법을 다 시도하는 것 자체가 함정이야.** 레짐 방법 6종 × 레짐 개수 2~3 × 피처 조합 여러 개를 다 돌려보고 결과가 유의한 것만 보고하면, 12번의 선택 편향이 극단적으로 커져. 흔히 "갈림길의 정원(garden of forking paths)"이라고 불러. **주 분석 방법을 미리 정하고, 나머지는 강건성 확인용으로** 써야 해.

추천하는 구조는 이래.

| 역할 | 방법 | 이유 |
|---|---|---|
| **주 분석 1** | 제도적 레짐 (공매도 금지) | 외생적, 순환성 없음, 한국 시장만의 기여점 |
| **주 분석 2** | JM 또는 HMM 시장 레짐 (KOSPI, 필터링) | 반복되는 변동성 국면 포착 |
| **강건성** | VKOSPI 기반 STR 또는 고정 임계값 | 관측 가능한 레짐으로도 결과가 유지되나 |
| **모형 결합** | 방식 ② 상호작용항 | δ 검정으로 연구 질문에 직접 답함 |

**결과 해석의 원칙:** 레짐 효과(δ ≠ 0)가 **여러 레짐 정의에서 일관되게** 나올 때만 "레짐에 따라 시그널이 다르다"고 주장할 수 있어. 한 방법에서만 나오면, 그건 시장의 성질이 아니라 그 방법의 산물일 가능성이 커.

---

## 계산 코드

```python
import numpy as np
import statsmodels.api as sm
from scipy.optimize import least_squares

# (1) TAR: 격자 탐색으로 임계값 c 추정
def fit_tar(y, x, q, trim=0.15):
    cands = np.quantile(q, np.linspace(trim, 1 - trim, 50))
    best = (None, np.inf)
    for c in cands:
        D = (q > c).astype(float)
        X = np.column_stack([np.ones_like(x), x, D, x * D])   # 상호작용 형태
        res = sm.OLS(y, X).fit()
        if res.ssr < best[1]:
            best = (c, res.ssr)
    return best[0]
# 주의: c의 유의성은 표준 t검정이 아니라 Hansen 부트스트랩 검정으로 판단해야 함

# (2) LSTR: 비선형 최소제곱
def lstr_resid(params, y, x, q):
    a1, b1, a2, b2, log_g, c = params
    G = 1 / (1 + np.exp(-np.exp(log_g) * (q - c)))   # γ > 0 보장
    yhat = (a1 + b1 * x) * (1 - G) + (a2 + b2 * x) * G
    return y - yhat

p0 = [0, 0, 0, 0, np.log(1.0), np.median(q)]
lstr = least_squares(lstr_resid, p0, args=(y, x, q))

# (3) 방식 ②: 상호작용항 회귀 (레짐 확률이나 더미 모두 가능)
D = regime_prob                                          # 필터링 확률 또는 0/1 더미
X = sm.add_constant(np.column_stack([x, D, x * D]))
res = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})
print(res.summary())   # x*D 계수(δ)의 t값이 연구 질문에 대한 답
```

---

**한 줄 요약:** 레짐 전환 방법은 "레짐을 어떻게 정하나"(관측 변수 임계값 / 부드러운 전환 / 숨겨진 확률 레짐 / 군집 / 구조적 변화 / 규칙·제도)와 "레짐을 모형에 어떻게 넣나"(표본 분할 / 상호작용항 / 동시 추정)의 조합이야. 네 프로젝트에는 **제도적 레짐과 시장 레짐을 주 분석으로 미리 정하고, 상호작용항의 δ를 검정**하는 구조가 가장 깔끔해. 여러 방법은 강건성 확인에만 써야 해.

다음 15번 **레짐의 정의**는 지금 본 방법들로 **"무엇을" 레짐이라 부를 것인가**, 즉 변동성 레짐인지, 추세 레짐인지, 유동성 레짐인지를 정하는 문제야. 준비되면 말해줘.